# Tutorial Silsilah Keluarga UnifyWeaver

Buku catatan interaktif ini mendemonstrasikan cara menggunakan UnifyWeaver untuk mengompilasi predikat Prolog ke dalam skrip Bash.

## Prasyarat

- SWI-Prolog terpasang
- Pustaka UnifyWeaver tersedia
- Kernel Jupyter Prolog terpasang (`pip install prolog-jupyter-kernel`)

## Tujuan Pembelajaran

Di akhir buku catatan ini, Anda akan mampu:
1. Mendefinisikan fakta dan aturan Prolog
2. Menggunakan UnifyWeaver untuk mengompilasi predikat ke Bash
3. Menguji skrip Bash yang dihasilkan
4. Memahami kompilasi penutupan transitif (transitive closure)

## Langkah 1: Inisialisasi Lingkungan UnifyWeaver

Pertama, kita perlu memuat modul UnifyWeaver. Kita akan menggunakan file `init.pl` dari direktori education.

In [ ]:
% Muat file inisialisasi
['../init'].

## Langkah 2: Mendefinisikan Hubungan Keluarga

Mari kita definisikan beberapa hubungan orang tua-anak dari silsilah keluarga alkitabiah.

In [ ]:
% Definisikan fakta-fakta parent
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## Langkah 3: Menguji Kueri Orang Tua

Sebelum mengompilasi, mari kita verifikasi bahwa data kita benar dengan beberapa kueri Prolog.

In [ ]:
% Kueri: Siapa anak-anak Abraham?
parent(abraham, Child).

In [ ]:
% Kueri: Siapa anak-anak Yakub?
parent(jacob, Child).

## Langkah 4: Mendefinisikan Hubungan Leluhur

Sekarang kita definisikan penutupan transitif — relasi `ancestor`.

In [ ]:
% Definisikan ancestor sebagai penutupan transitif dari parent
:- dynamic ancestor/2.

% Kasus dasar: orang tua adalah leluhur
ancestor(X, Y) :- parent(X, Y).

% Kasus rekursif: jika X adalah orang tua dari Y dan Y adalah leluhur dari Z, maka X adalah leluhur dari Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## Langkah 5: Menguji Kueri Leluhur

Mari kita verifikasi bahwa predikat leluhur kita berfungsi dengan benar.

In [ ]:
% Kueri: Apakah Abraham adalah leluhur Yakub?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Kueri: Siapa semua keturunan Abraham?
ancestor(abraham, Descendant).

## Langkah 6: Mengompilasi Parent ke Bash

Sekarang bagian yang menyenangkan — mari kita kompilasi fakta `parent/2` kita ke dalam skrip Bash!

In [ ]:
% Muat kompilator stream
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Kompilasi fakta-fakta parent ke bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## Langkah 7: Menyimpan Skrip Parent

Mari kita simpan kode Bash yang dihasilkan ke dalam file.

In [ ]:
% Simpan ke file
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## Langkah 8: Mengompilasi Ancestor ke Bash

Sekarang kompilasi predikat `ancestor/2`, yang menggunakan rekursi.

In [ ]:
% Muat kompilator rekursif
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Kompilasi ancestor ke bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## Langkah 9: Menyimpan Skrip Ancestor

Simpan skrip leluhur ke dalam file.

In [ ]:
% Simpan ke file
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## Langkah 10: Menguji Skrip yang Dihasilkan

Sekarang uji skrip Bash yang dihasilkan! Kita akan menggunakan sihir `%%bash` untuk menjalankan perintah bash.

In [ ]:
%%bash
# Muat skrip parent dengan source
source ../output/parent.sh

# Uji: Siapa anak-anak Abraham?
echo "Anak-anak Abraham:"
parent abraham

In [ ]:
%%bash
# Muat kedua skrip dengan source
source ../output/parent.sh
source ../output/ancestor.sh

# Uji: Siapa keturunan Abraham?
echo "Keturunan Abraham:"
ancestor abraham

In [ ]:
%%bash
# Muat kedua skrip dengan source
source ../output/parent.sh
source ../output/ancestor.sh

# Uji: Apakah Abraham adalah leluhur Yehuda?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Ya, Abraham adalah leluhur Yehuda"
else
    echo "✗ Tidak"
fi

## Langkah 11: Memahami Strategi Kompilasi

Mari kita analisis apa yang dilakukan UnifyWeaver:

1. **Kompilasi parent**: Menggunakan `stream_compiler` untuk membuat fungsi streaming sederhana yang memancarkan semua pasangan orang tua-anak

2. **Kompilasi ancestor**: Mendeteksi pola penutupan transitif dan menerapkan optimasi BFS (pencarian melebar) untuk menghitung secara efisien semua leluhur yang dapat dijangkau

Mari kita verifikasi strategi kompilasi:

In [ ]:
% Periksa apakah ancestor diklasifikasikan sebagai rekursif
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## Ringkasan

Dalam buku catatan ini, Anda telah mempelajari:

✅ Cara mendefinisikan fakta dan aturan Prolog

✅ Cara menggunakan `stream_compiler` UnifyWeaver untuk fakta

✅ Cara menggunakan `recursive_compiler` UnifyWeaver untuk predikat rekursif

✅ Cara menguji skrip Bash yang dihasilkan

✅ Bahwa UnifyWeaver secara otomatis mendeteksi penutupan transitif dan menerapkan optimasi BFS

## Langkah Selanjutnya

Cobalah latihan-latihan ini:

1. Tambahkan lebih banyak anggota keluarga ke silsilah
2. Definisikan predikat `grandparent/2` dan kompilasikan
3. Buat predikat `sibling/2` (dua orang dengan orang tua yang sama)
4. Jelajahi kode Bash yang dihasilkan untuk memahami algoritma BFS

Lanjutkan ke **Buku Catatan 2: Perbandingan Pola Rekursi** untuk mempelajari pola rekursi tingkat lanjut!